# vi-function-calling-slm · 04 xuất GGUF + đánh giá

Cần **GPU T4**, bật Internet. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input: code + dữ liệu (notebook 01) + output notebook 03 (`sft-merged`, `dpo-merged`); trên Colab nằm sẵn trên Drive.
Secrets (đều tuỳ chọn): `HF_TOKEN` (đẩy GGUF), `GEMINI_API_KEY` (cận trên, chỉ để đánh giá).
Không cần Unsloth. Ngân sách ~40 phút trên Colab T4: phần nào hết giờ thì bỏ qua và in ra.

1. Xuất SFT+DPO sang GGUF `q4_k_m` (~1.1GB) bằng llama.cpp
2. Đánh giá trên bộ viết tay (đủ 45 câu) + mẫu 30 câu của tập test tổng hợp:
   - SFT+DPO, SFT, Qwen3-1.7B gốc (chỉ prompt, cùng system prompt + tools) — transformers fp16 trên GPU
   - SFT+DPO GGUF Q4_K_M qua llama.cpp (CPU) — bản thật sự chạy trên laptop, để xem lượng tử hoá mất bao nhiêu
   - Gemini (function calling qua API) làm cận trên
3. Bảng `results/eval.md`

Độ trễ ở đây đo trên CPU Kaggle/Colab, không đại diện cho laptop (đo lại bằng `vi_fc.evaluate predict` trên máy thật).

In [ ]:
import gc, glob, json, os, sys, time

T0 = time.time()


BUDGET = 40  # phút; không có Drive phải gộp lại 2 adapter từ HF (~8 phút) nên dư hơn bản cũ (20)


def left(budget=BUDGET):
    return budget - (time.time() - T0) / 60  # số phút còn lại của ngân sách


IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_fc/__init__.py", recursive=True)[0].rsplit("/vi_fc", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-function-calling-slm" if IN_COLAB else os.path.abspath("vi-function-calling-slm")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-function-calling-slm {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    WORK = os.getcwd()
    if IN_COLAB:
        from google.colab import drive
        try:  # không ai bấm cho phép Drive thì sau 2 phút chạy trên /content, output trước lấy từ HF Hub
            drive.mount("/content/drive", timeout_ms=120_000)
            WORK = "/content/drive/MyDrive/ai-portfolio/vi-function-calling-slm"
        except Exception as e:
            print("không mount được Drive, chạy trên /content/work:", e)
            WORK = "/content/work"
NO_DRIVE = WORK == "/content/work"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


# dữ liệu notebook 01 có sẵn trong data/gen của repo. Chưa có bản merge của notebook 03 (không có Drive, hoặc
# Drive không chứa) -> tải 2 adapter từ HF Hub rồi gộp lại (~5 phút), ghi ra ổ local của runtime cho nhanh
from vi_fc.hub import pull

os.environ.setdefault("HF_TOKEN", secret("HF_TOKEN") or "")
M = f"{REPO}/merged"  # find() có tìm trong REPO
if not glob.glob(f"{WORK}/dpo-merged/config.json") and not os.path.exists(f"{M}/dpo-merged/config.json"):
    !pip install -q peft
    # peft mới báo lỗi với torchao cài sẵn trên Colab (ImportError version); peft không cần torchao -> gỡ
    !pip uninstall -q -y torchao
    from vi_fc.train import merge_lora
    pull("vi-fc-qwen3-1.7b", f"{M}/vi-fc-sft-lora", subdir="sft-lora")
    pull("vi-fc-qwen3-1.7b", f"{M}/vi-fc-dpo-lora", subdir="dpo-lora")
    merge_lora(f"{M}/vi-fc-sft-lora", f"{M}/sft-merged")
    merge_lora(f"{M}/vi-fc-dpo-lora", f"{M}/dpo-merged", base=f"{M}/sft-merged")

DATA = os.path.dirname(find("data/gen/test.jsonl", "chạy notebook 01 trước"))
SFT = os.path.dirname(find("sft-merged/config.json", "chạy notebook 03 trước"))
DPO = os.path.dirname(find("dpo-merged/config.json", "chạy notebook 03 trước"))
print(CODE, DATA, SFT, DPO)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip list --format=freeze | grep -iE "^(torch|torchvision|torchaudio)==" > /tmp/keep-torch.txt
!pip install -q -c /tmp/keep-torch.txt accelerate
# chỉ lấy wheel CPU dựng sẵn: build từ source mất ~15 phút, quá ngân sách -> không có wheel thì bỏ eval GGUF
!pip install -q --only-binary llama-cpp-python llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu || echo "không có wheel llama-cpp-python cho runtime này"

for k in ("HF_TOKEN", "GEMINI_API_KEY", "HF_USER"):
    v = secret(k)
    if v:
        os.environ[k] = v
    else:
        print("thiếu secret", k)

## Xuất GGUF q4_k_m
convert_hf_to_gguf.py của llama.cpp ra f16, rồi lượng tử hoá bằng llama.cpp có sẵn trong llama-cpp-python
(không có thì build riêng `llama-quantize`). Lỗi thì bỏ qua GGUF, phần eval HF vẫn chạy.

In [ ]:
import shutil, subprocess

LLAMA = "/tmp/llama.cpp"
GGUF = f"{WORK}/gguf/vi-fc-qwen3-1.7b-Q4_K_M.gguf"


def sh(*cmd):
    # subprocess không in ra cell -> bắt output, lỗi thì đưa đoạn cuối vào exception
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{' '.join(cmd)[:120]} lỗi:\n" + (r.stderr or r.stdout)[-2000:])


def quantize(f16, out):
    try:
        import ctypes, llama_cpp

        llama_cpp.llama_backend_init()
        p = llama_cpp.llama_model_quantize_default_params()
        p.ftype = llama_cpp.LLAMA_FTYPE_MOSTLY_Q4_K_M
        p.nthread = os.cpu_count()
        if llama_cpp.llama_model_quantize(f16.encode(), out.encode(), ctypes.byref(p)) == 0:
            return
        print("llama_model_quantize trả mã lỗi")
    except Exception as e:
        print("quantize qua llama-cpp-python lỗi:", e)
    if left() < 10:
        raise RuntimeError("không đủ thời gian build llama-quantize (~5 phút)")
    sh("bash", "-c", f"cmake -S {LLAMA} -B {LLAMA}/build -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF"
                     f" && cmake --build {LLAMA}/build --target llama-quantize -j")
    sh(f"{LLAMA}/build/bin/llama-quantize", f16, out, "Q4_K_M")


def export_gguf():
    if os.path.exists(GGUF) and os.path.getmtime(GGUF) > os.path.getmtime(f"{DPO}/config.json"):
        return GGUF  # đã xuất từ đúng bản dpo-merged này
    if not os.path.exists(LLAMA):
        sh("git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA)
    f16, q4 = "/tmp/vi-fc-f16.gguf", "/tmp/vi-fc-q4km.gguf"
    sh(sys.executable, f"{LLAMA}/convert_hf_to_gguf.py", DPO, "--outtype", "f16", "--outfile", f16)
    quantize(f16, q4)
    os.remove(f16)
    os.makedirs(os.path.dirname(GGUF), exist_ok=True)
    shutil.move(q4, GGUF)  # ghi ở /tmp rồi mới chuyển: lỗi giữa chừng không để lại file dở trên Drive
    return GGUF


try:
    GGUF = export_gguf()
    print(GGUF, f"{os.path.getsize(GGUF) / 2**30:.2f} GB, còn {left():.0f} phút")
except Exception as e:
    print("xuất GGUF lỗi, bỏ qua phần GGUF (eval HF bên dưới vẫn chạy):", e)
    GGUF = None

## Đánh giá
Bộ viết tay luôn chạy đủ 45 câu; tập test tổng hợp lấy mẫu cố định 30 câu cho vừa 20 phút. Hệ thống nào hết giờ thì
bỏ qua (in ra), chạy lại notebook thì chỉ chạy phần còn thiếu. Muốn đủ cả tập test: tăng `N_SYNTH`.

In [ ]:
import random

import torch
from vi_fc.evaluate import load_jsonl, predict, report
from vi_fc.inference import FunctionCaller

N_SYNTH = 30
test = load_jsonl(f"{DATA}/test.jsonl")
SETS = {"manual": load_jsonl(f"{REPO}/data/test_manual.jsonl"),
        "synth_test": random.Random(0).sample(test, min(N_SYNTH, len(test)))}
os.makedirs("results/preds", exist_ok=True)
print({k: len(v) for k, v in SETS.items()})


def done(path, n, src):
    # đủ dòng và mới hơn model -> không chạy lại
    return (os.path.exists(path) and sum(1 for _ in open(path, encoding="utf-8")) == n
            and (src is None or os.path.getmtime(path) > os.path.getmtime(src)))


def run(system, make_fc, src=None, sets=SETS, min_left=2.0, sleep_s=0.0):
    todo = {n: r for n, r in sets.items() if not done(f"results/preds/{system}_{n}.jsonl", len(r), src)}
    if not todo:
        print("đã có", system)
        return
    if left() < min_left:
        print(f"còn {left():.1f} phút, bỏ qua {system} {list(todo)}")
        return
    fc = make_fc()
    for name, recs in todo.items():
        predict(recs, fc, system, name, f"results/preds/{system}_{name}.jsonl", sleep_s)
        print(f"xong {system} {name}, còn {left():.0f} phút")


# transformers fp16 trên GPU; bản quan trọng chạy trước, model gốc chỉ chạy nếu còn giờ
for system, path, min_left in (("sft-dpo", DPO, 2), ("sft", SFT, 8), ("qwen3-1.7b-base", "Qwen/Qwen3-1.7B", 8)):
    src = f"{path}/config.json" if os.path.isdir(path) else None
    run(system, lambda p=path: FunctionCaller("hf", p, guard=False, max_tokens=128), src, min_left=min_left)
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# GGUF qua llama.cpp trên CPU: đúng đường chạy trên laptop
try:
    import llama_cpp  # noqa: F401
except Exception as e:
    print("không có llama-cpp-python (không có wheel cho runtime này?), bỏ qua eval GGUF:", e)
    GGUF_EVAL = None
else:
    GGUF_EVAL = GGUF


def gguf_fc():
    fc = FunctionCaller("llama_cpp", GGUF_EVAL, n_threads=os.cpu_count(), guard=False, max_tokens=128)
    print("warmup", fc.warmup())
    return fc


if GGUF_EVAL:
    run("sft-dpo-q4km", gguf_fc, GGUF_EVAL, sets={"manual": SETS["manual"]})
    run("sft-dpo-q4km", gguf_fc, GGUF_EVAL, sets={"synth_test": SETS["synth_test"]}, min_left=4)

In [ ]:
if os.environ.get("GEMINI_API_KEY") and left() > 5:
    gem = FunctionCaller("gemini", guard=False)
    try:
        gem.decide("bật điều hoà 22 độ")  # thử 1 câu trước: sai key / tên model thì bỏ luôn, khỏi chờ cả bộ
    except Exception as e:
        print("Gemini lỗi, bỏ qua:", e)
    else:
        run("gemini", lambda: gem, sets={"manual": SETS["manual"]}, sleep_s=4.0)  # free tier ~15 request/phút
        run("gemini", lambda: gem, sets={"synth_test": SETS["synth_test"]}, min_left=4, sleep_s=4.0)

In [ ]:
res = report(sorted(glob.glob("results/preds/*.jsonl")), "results")
print(open("results/eval.md", encoding="utf-8").read())
print(f"tổng thời gian {BUDGET - left():.0f} phút")

In [ ]:
if GGUF:
    from vi_fc.hub import push

    push(GGUF, "vi-fc-qwen3-1.7b-GGUF", path_in_repo="vi-fc-qwen3-1.7b-Q4_K_M.gguf")  # thiếu token / lỗi thì chỉ in ra

## Câu sai của bản GGUF trên bộ viết tay

In [ ]:
from vi_fc.match import actions_match
from vi_fc.parse import Action

if os.path.exists("results/preds/sft-dpo-q4km_manual.jsonl"):
    for r in load_jsonl("results/preds/sft-dpo-q4km_manual.jsonl"):
        if not actions_match(Action.from_dict(r["gold"]), Action.from_dict(r["pred"])):
            print(r["id"], r["user"]); print("   gold:", r["gold"]); print("   raw :", r["raw"][:200])